In [1]:
from pathlib import Path
import numpy as np
from lisaorbits import OEMOrbits
# ============================================================
# Explicit settings
# ============================================================

BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

ORBITS_FILE = INPUT_DIR / "orbits.h5"

# Choose one:
#   "esa-trailing" = constellation trails Earth by 20 degrees
#   "esa-leading"  = constellation leads Earth by 20 degrees
ORBIT_NAME = "esa-trailing"

# Orbit-file sampling.
# This does NOT need to be your science sampling dt.
# It is the sampling grid stored in the orbit HDF5 file.
DT_ORBIT = 10_000.0      # seconds
DURATION = 30 * 24 * 3600.0   # 30 days
SIZE = int(DURATION / DT_ORBIT) + 1


def main():
    print(f"Creating realistic LISA orbit file using {ORBIT_NAME}")

    # This uses ESA-provided OEM orbits included/downloaded by lisaorbits.
    # The underlying spacecraft positions are interpolated from OEM files.
    orbits = OEMOrbits.from_included(
        ORBIT_NAME,
        version="latest",
    )

    print(f"ESA orbit starts at t_start = {orbits.t_start:.3f} s")
    print(f"ESA orbit ends   at t_end   = {orbits.t_end:.3f} s")

    # Use the default initial time of the OEM orbit.
    # The docs say OEMOrbits uses t_start + 10 s by default
    # to leave margin for light-travel-time calculations.
    t0 = "init"

    if ORBITS_FILE.exists():
        ORBITS_FILE.unlink()

    orbits.write(
        str(ORBITS_FILE),
        t0=t0,
        dt=DT_ORBIT,
        size=SIZE,
        mode="x",
    )

    print(f"Wrote orbit file:")
    print(f"  {ORBITS_FILE}")

    # Quick sanity check on light travel times
    t = np.linspace(orbits.t_init, orbits.t_init + 5 * DT_ORBIT, 6)
    links = [12, 23, 31, 13, 32, 21]

    ltt = orbits.compute_ltt(t, links)
    ltt_dot = orbits.compute_ltt_derivative(t, links)

    print("\nExample light-travel times [s]:")
    print(ltt)

    print("\nExample light-travel-time derivatives [s/s]:")
    print(ltt_dot)


if __name__ == "__main__":
    main()

Creating realistic LISA orbit file using esa-trailing


  0%|                                              | 0.00/85.1k [00:00<?, ?B/s]

  0%|                                              | 0.00/85.2k [00:00<?, ?B/s]

  0%|                                              | 0.00/85.3k [00:00<?, ?B/s]

OEM preferred interpolation method ignored, using spline interpolation (see InterpolatedOrbits for details)


ESA orbit starts at t_start = 61171229.328 s
ESA orbit ends   at t_end   = 400424722.588 s
Wrote orbit file:
  /Users/shubhanshutiwari/Desktop/LISA-TF/lisa_link_sim/inputs/orbits.h5

Example light-travel times [s]:
[[8.20291744 8.35366614 8.25835077 8.25671349 8.3543685  8.20384048]
 [8.20316903 8.35376163 8.25821513 8.25657756 8.35446695 8.20408939]
 [8.20342116 8.35385657 8.25807951 8.25644165 8.35456486 8.20433884]
 [8.20367382 8.35395097 8.25794392 8.25630578 8.35466222 8.20458881]
 [8.203927   8.35404482 8.25780836 8.25616995 8.35475903 8.20483931]
 [8.20418072 8.35413812 8.25767283 8.25603415 8.35485529 8.20509034]]

Example light-travel-time derivatives [s/s]:
[[ 2.51323058e-08  9.57593459e-09 -1.35654281e-08 -1.35951188e-08
   9.87302645e-09  2.48649064e-08]
 [ 2.51857605e-08  9.52145595e-09 -1.35630386e-08 -1.35920709e-08
   9.81826811e-09  2.49179823e-08]
 [ 2.52390773e-08  9.46690461e-09 -1.35604613e-08 -1.35888349e-08
   9.76343589e-09  2.49709213e-08]
 [ 2.52922557e-08  9.